# Транспорт с kernel bypass: анализ замеров

Разбор end-to-end задержки транспорта DPDK/ENA между `producer` и `consumer`.
Идеи и решения — в [harness/SOLUTION.md](harness/SOLUTION.md), все числа с
оговорками — в [harness/RESULTS.md](harness/RESULTS.md), запуск — в
[harness/RUN.md](harness/RUN.md).

## Стенд

| | |
|---|---|
| инстансы | AWS c6in, одна подсеть VPC, placement group: отправитель `c6in.2xlarge`, приёмники `c6in.2xlarge` + 3x `c6in.xlarge` |
| сеть | ENI под DPDK (`igb_uio`, `wc_activate=1`) + управляющая `ens5` |
| DPDK | 25.11.2, ENA PMD, `llq_policy=3` (large LLQ) |
| ядра | producer и отправитель на изолированных, SMT-сиблинги погашены |

## Как читать числа

Задержка режется на три отрезка метками в заголовке кадра:

| | |
|---|---|
| **a** | producer → отправитель, включая ожидание набора датаграммы |
| **b** | провод: TX, сеть, RX |
| **c** | приёмник → consumer |

`a` и `c` целиком внутри одного хоста и потому точны. `b` — единственный,
пересекающий границу хостов, и единственный, несущий расхождение часов.
`chrony` остановлен на весь сеанс, смещение снимается пинг-понгом до каждого
приёмника отдельно до и после прогона, поправка интерполируется **по моментам
замеров**. Уход часов на стенде — 0.011…0.578 ppm, у каждой машины свой.

**Разброс среды между прогонами одного и того же кода доходит до 20%.**
Разницы меньше этого по одиночным прогонам не интерпретируются.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from bokeh.io import output_notebook, show
from bokeh.layouts import column, gridplot
from bokeh.models import ColumnDataSource, HoverTool, NumeralTickFormatter, Span, Label
from bokeh.palettes import Category10
from bokeh.plotting import figure

output_notebook()

BLUE, RED, GREEN, GREY = '#2a78d6', '#d64545', '#2aa855', '#888888'
PCTS = ['p50', 'p99', 'p99.9', 'p99.99']

def fig(title, x_axis_label, y_axis_label, y_log=False, **kw):
    p = figure(title=title, x_axis_label=x_axis_label, y_axis_label=y_axis_label,
               height=380, width=760, toolbar_location='above',
               y_axis_type='log' if y_log else 'linear', **kw)
    p.yaxis.formatter = NumeralTickFormatter(format='0,0')
    p.xaxis.formatter = NumeralTickFormatter(format='0,0')
    return p

Loading BokehJS ...

## Данные

Числа сняты с прогонов и внесены сюда явно, чтобы ноутбук считался без
наличия сырых CSV. Раздел «Распределение целиком» ниже читает CSV, если он
есть рядом.

Все прогоны ниже — **ноль потерь**, `gate_gaps` = 0, `gate_missing` = 0,
если не указано иное.

In [2]:
# Один получатель, развёртка по темпу. pack = min_pack подобран так, чтобы
# удержать ПАКЕТНЫЙ поток примерно постоянным.
sweep = pd.DataFrame([
    # rate,   pack,     pps,   a50,  b50, c50,   p50,   p99, p99.9, p99.99
    (  100_000, 1,   99_999,  186, 6305, 202,  6732, 11455, 15562, 21042),
    (  200_000, 1,  199_999,  186, 6571, 204,  6993, 13143, 15744, 22074),
    (  600_000, 4,  150_060, 2919, 5451, 338,  9058, 16053, 19230, 23127),
    (  800_000, 6,  133_333, 3543, 7681, 404, 11778, 19412, 22373, 27103),
    (1_000_000, 8,  125_000, 4117, 7095, 464, 11565, 19368, 22438, 28384),
    (1_000_000, 5,  200_000, 2185, 6394, 370,  8823, 15620, 18744, 25268),
], columns=['rate','pack','pps','a50','b50','c50','p50','p99','p99.9','p99.99'])

# Лучшая конфигурация на каждый темп (для 1M это min_pack=5).
best = sweep.sort_values('p50').groupby('rate', as_index=False).first().sort_values('rate')

# То же до батчинга: пакет на каждое сообщение.
nobatch = pd.DataFrame([
    (  200_000,   7117,   7516,  12167,  15443,  30941, 0.0),
    (  600_000,  17842,  18367,  27704,  37766,  62624, 0.0),
    (  800_000,  38451,  39198,  59727,  89000, 111794, 0.0),
    (1_000_000, 671712, 672637, 733823, 748946, 763864, 16.2),
], columns=['rate','b50','p50','p99','p99.9','p99.99','loss_pct'])

# Baseline: ядерный UDP (sendmmsg), решение агента, тот же стенд.
baseline = pd.DataFrame([
    (  200_000,     9967,    15188,    22021,    38916, 0.0),
    (  600_000,    24815,    37342,    59606,   109140, 0.0),
    (  800_000,    33735,    51068,    79817,   129502, 0.0),
    (1_000_000, 11013143, 11336409, 11583937, 11722339, 1.02),
], columns=['rate','p50','p99','p99.9','p99.99','loss_pct'])

best[['rate','pack','pps','p50','p99','p99.9','p99.99']]

,rate,pack,pps,p50,p99,p99.9,p99.99
0,100000,1,99999,6732,11455,15562,21042
1,200000,1,199999,6993,13143,15744,22074
2,600000,4,150060,9058,16053,19230,23127
3,800000,6,133333,11778,19412,22373,27103
4,1000000,5,200000,8823,15620,18744,25268


In [3]:
# Веер: 1M msg/s на 4 получателя одновременно (--pack 8 --min-pack 4).
# 4 млн сообщений в секунду суммарно, 999 385 pkt/s, ноль потерь.
fanout = pd.DataFrame([
    ('.83',  0.011, 1573, 16167, 487, 18461, 31475, 38842, 54128),
    ('.60', -0.142, 1573, 14094, 398, 16244, 25556, 30167, 56259),
    ('.149', 0.091, 1573, 16607, 411, 18822, 28251, 33145, 62249),
    ('.120', 0.578, 1573, 17740, 468, 19991, 33042, 40754, 58809),
], columns=['peer','drift_ppm','a50','b50','c50','p50','p99','p99.9','p99.99'])

# То же 1M x 4, но MIN_PACK=8: вдвое меньше пакетов (500k против 999k).
fanout8 = pd.DataFrame([
    ('.83',  4123, 8676, 466, 13471, 21972, 25921, 71831),
    ('.60',  4123, 8142, 464, 12944, 20881, 23698, 71104),
    ('.149', 4123, 8774, 478, 13491, 20892, 24591, 98843),
    ('.120', 4123, 10050, 445, 14375, 22540, 27635, 87785),
], columns=['peer','a50','b50','c50','p50','p99','p99.9','p99.99'])

# И MIN_PACK=16 на MTU 9001: 250k pkt/s. Отрезок a вдвое больше, чем у
# MIN_PACK=8, а p99.99 вдвое ЛУЧШЕ -- значит хвост там делала не упаковка.
fanout16 = pd.DataFrame([
    ('.83',  7406,  9180, 722, 18118, 27930, 31207, 41971),
    ('.60',  7406,  6931, 714, 15760, 25009, 27423, 30388),
    ('.149', 7406, 10118, 740, 18734, 27742, 30361, 36514),
    ('.120', 7406,  8210, 700, 16953, 26533, 30127, 41165),
], columns=['peer','a50','b50','c50','p50','p99','p99.9','p99.99'])

# Веер 600k x 4 (2.4M msg/s суммарно), три конфигурации. Первая -- с занижённым
# PACK_WAIT_SPINS, из-за чего батчинг не работал; диапазоны по 4 получателям.
fanout600 = pd.DataFrame([
    ('wait 1000\nPACK 4..12',        2.15, 1_110_000,  1961, 21559, 24379, 32266, 39523, 36757, 46759),
    ('wait 10000\nPACK 12, jumbo',  12.00,   200_136, 10109, 16932, 19923, 28339, 31186, 31486, 35037),
    ('wait 10000\nPACK 8, MTU 1500', 8.00,   300_119,  6780, 13358, 15266, 21660, 24294, 24926, 30331),
], columns=['config','frames_per_dgram','pps','a50',
            'p50_lo','p50_hi','p99_lo','p99_hi','p999_lo','p999_hi'])

fanout600

,peer,drift_ppm,a50,b50,c50,p50,p99,p99.9,p99.99
0,.83,0.011,1573,16167,487,18461,31475,38842,54128
1,.60,-0.142,1573,14094,398,16244,25556,30167,56259
2,.149,0.091,1573,16607,411,18822,28251,33145,62249
3,.120,0.578,1573,17740,468,19991,33042,40754,58809


## 1. Задержка против темпа: три транспорта

Логарифмическая ось Y — иначе точка 1M без батчинга (673 мкс) и baseline
(11 мс) сплющивают всё остальное в линию.

In [4]:
p = fig('e2e p50 против темпа (один получатель)', 'сообщений в секунду',
        'наносекунды, log', y_log=True)

for df, name, colour, dash in [
    (best,     'DPDK + батчинг',      BLUE,  'solid'),
    (nobatch,  'DPDK, пакет=сообщение', GREEN, 'dashed'),
    (baseline, 'baseline: ядерный UDP', RED,   'dotted'),
]:
    src = ColumnDataSource(df)
    p.line('rate', 'p50', source=src, line_width=2.5, color=colour,
           line_dash=dash, legend_label=name)
    p.scatter('rate', 'p50', source=src, size=9, color=colour, legend_label=name)

p.add_tools(HoverTool(tooltips=[('темп', '@rate{0,0} msg/s'), ('p50', '@p50{0,0} нс')]))
p.legend.location = 'top_left'
p.legend.click_policy = 'hide'
show(p)

**Что видно.** До батчинга задержка растёт с темпом и на 1M msg/s срывается:
673 мкс при 16.2% потерь — отправитель упирается в пакетный шейпер среды
(1.081M pkt/s на пару хостов) и выбрасывает пакеты на полном TX-кольце.

С батчингом кривая почти плоская: **8.8 мкс на миллионе сообщений в секунду**
вместо 673, потери нулевые. Это 76x.

Baseline (ядерный UDP) на своём потолке (~740k датаграмм/с) срывается так же,
только раньше и в 11 мс.

## 2. Куда уходит задержка: разложение на отрезки

Только наше решение с батчингом. Столбцы — медианы отрезков.

In [5]:
labels = [f"{r//1000}k\npack {p}" for r, p in zip(sweep.rate, sweep.pack)]
src = ColumnDataSource(dict(x=labels, a=sweep.a50, b=sweep.b50, c=sweep.c50,
                            total=sweep.a50 + sweep.b50 + sweep.c50))

p = figure(x_range=labels, title='Из чего складывается e2e p50', height=400, width=760,
           x_axis_label='темп и упаковка', y_axis_label='наносекунды',
           toolbar_location='above')
p.vbar_stack(['a','b','c'], x='x', width=0.6, source=src,
             color=[GREEN, BLUE, GREY],
             legend_label=['a: producer -> отправитель (ожидание набора)',
                           'b: провод',
                           'c: приёмник -> consumer'])
p.yaxis.formatter = NumeralTickFormatter(format='0,0')
p.legend.location = 'top_left'
p.y_range.start = 0
show(p)

**Что видно.** `b` — задержка провода — держится в 5.5–7.7 мкс на **всём**
десятикратном диапазоне темпа. Она перестала зависеть от темпа сообщений,
потому что батчинг удерживает пакетный поток почти постоянным, а шейпит среда
именно пакеты.

Растёт только `a`, и это осознанная плата: ожидание, пока наберётся `min_pack`
кадров. Две последние пары столбцов — один и тот же темп 1M msg/s при
`min_pack` 8 и 5; видно, что лишнее ожидание убирается настройкой.

## 3. Ключевое: `b` зависит от ПАКЕТНОГО потока, а не от темпа сообщений

Тот же набор прогонов, но по оси X — пакеты в секунду.

In [6]:
src = ColumnDataSource(dict(pps=sweep.pps, b50=sweep.b50, rate=sweep.rate, pack=sweep.pack))
p = fig('b p50 против пакетного потока', 'пакетов в секунду', 'наносекунды')
p.scatter('pps', 'b50', source=src, size=13, color=BLUE)
p.add_tools(HoverTool(tooltips=[('pkt/s', '@pps{0,0}'), ('b p50', '@b50{0,0} нс'),
                                ('темп', '@rate{0,0} msg/s'), ('pack', '@pack')]))

lo, hi = sweep.b50.min(), sweep.b50.max()
for y, dash in [(lo, 'dashed'), (hi, 'dashed')]:
    p.add_layout(Span(location=y, dimension='width', line_color=GREY, line_dash=dash))
p.add_layout(Label(x=105_000, y=hi + 250, text=f'разброс {lo:,}..{hi:,} нс — это шум среды',
                   text_color=GREY, text_font_size='10pt'))
p.y_range.start = 0
show(p)

**Что видно.** Порядка нет: минимум приходится на середину диапазона
(150k pkt/s → 5 451 нс), а не на край. Весь разброс 5.5–7.7 мкс укладывается
в шум среды.

Отсюда практическое правило: **опускать пакетный поток ниже ~200k pkt/s
бессмысленно** — `b` от этого не улучшается, а `a` продолжает расти линейно
по `min_pack`. Значит `min_pack ≈ rate / 200 000`.

Правило проверено предсказанием: при 1M msg/s оно даёт `min_pack` 5 вместо 8,
ожидалось `a` p50 ~2 500 нс и `e2e` p50 ~10 мкс — измерено **2 185** и **8 823**.
Улучшились все перцентили без исключения.

## 4. Проверка модели отрезка `a`

Внутри датаграммы кадр номер k ждёт `(min_pack - k) / rate`, поэтому `a`
распределён равномерно на `[0, min_pack/rate]`, а не хвостат. Отсюда
`a` p50 ≈ `min_pack / (2 x rate)`.

In [7]:
checks = pd.DataFrame([
    # min_pack,      rate,  измеренный a p50
    ( 4,   600_000, 2919),
    ( 6,   800_000, 3543),
    ( 8,   800_000, 4142),
    ( 8,   800_000, 4148),
    ( 8,   800_000, 5117),
    (10,   800_000, 5387),
    ( 5, 1_000_000, 2185),
    ( 8, 1_000_000, 4117),
    ( 8,   600_000, 6780),   # веер, 4 получателя
    (12,   600_000, 10109),  # веер, 4 получателя
], columns=['min_pack','rate','a50_measured'])
checks['a50_predicted'] = checks.min_pack / (2 * checks.rate) * 1e9
checks['ratio'] = checks.a50_measured / checks.a50_predicted

p = fig('Модель против измерения', 'предсказано, нс', 'измерено, нс')
lim = 11000
p.line([0, lim], [0, lim], color=GREY, line_dash='dashed', legend_label='идеальное совпадение')
p.scatter('a50_predicted', 'a50_measured', source=ColumnDataSource(checks),
          size=12, color=BLUE, legend_label='прогоны')
p.add_tools(HoverTool(tooltips=[('min_pack', '@min_pack'), ('темп', '@rate{0,0}'),
                                ('предсказано', '@a50_predicted{0,0}'),
                                ('измерено', '@a50_measured{0,0}'),
                                ('отношение', '@ratio{0.00}')]))
p.legend.location = 'top_left'
show(p)

print(f"отношение измеренного к предсказанному: "
      f"{checks.ratio.min():.2f}..{checks.ratio.max():.2f}, среднее {checks.ratio.mean():.2f}")

отношение измеренного к предсказанному: 0.83..1.03, среднее 0.91


**Что видно.** Восемь независимых прогонов ложатся на прямую. Модель
систематически чуть завышает (среднее отношение ~0.9): предсказание считает,
что кадры приходят строго равномерно, а producer выдаёт их пачками.

Практический смысл: **`a` можно посчитать заранее, не запуская прогон**, и
выбирать `min_pack` под требуемую задержку, а не подбирать перебором.

## 5. Веер: 1M msg/s на 4 получателя

4 млн сообщений в секунду суммарно, 999 385 pkt/s (полка — 1.081M),
ноль потерь у всех четверых.

In [8]:
p = figure(x_range=list(fanout.peer), title='Перцентили по получателям, 1M msg/s x 4',
           height=400, width=760, x_axis_label='получатель',
           y_axis_label='наносекунды', toolbar_location='above')

offsets = np.linspace(-0.27, 0.27, len(PCTS))
for (pct, colour, off) in zip(PCTS, Category10[4], offsets):
    p.vbar(x=[i + 0.5 + off for i in range(len(fanout))], top=fanout[pct],
           width=0.16, color=colour, legend_label=pct)
p.yaxis.formatter = NumeralTickFormatter(format='0,0')
p.legend.location = 'top_left'
p.y_range.start = 0
show(p)

print('уход часов у получателей, ppm:',
      ', '.join(f'{x:+.3f}' for x in fanout.drift_ppm))

уход часов у получателей, ppm: +0.011, -0.142, +0.091, +0.578


**Что видно.** Разброс между получателями невелик — p50 16.2…20.0 мкс при том,
что часы у четырёх машин расходятся с разной скоростью (от 0.011 до 0.578 ppm).
Именно поэтому смещение меряется **до каждого приёмника отдельно**: одна общая
поправка смешала бы разные расхождения в одну цифру.

Отрезок `a` у всех четверых совпадает до наносекунды (1 573 нс) — он снимается
один раз на сообщение, до размножения. Это заодно проверка, что поправка часов
не заехала в то, что измеряется на одном хосте.

До батчинга та же нагрузка была недостижима: 400k x 4 требовали 1.6M pkt/s при
полке 1.08M, 2.8% выбрасывалось на полном TX-кольце, задержка уходила в
миллисекунды.

### Влияние `MIN_PACK` при той же нагрузке

Тот же 1M msg/s на 4 получателя, но упаковка по 8 кадров вместо 4: пакетный поток
падает с 999 385 до 499 998 pkt/s.

In [ ]:
cmp = pd.DataFrame({
    'перцентиль': PCTS,
    'MIN_PACK=4  (999k pkt/s)': [fanout[k].mean()   for k in PCTS],
    'MIN_PACK=8  (500k pkt/s)': [fanout8[k].mean()  for k in PCTS],
    'MIN_PACK=16 (250k pkt/s)': [fanout16[k].mean() for k in PCTS],
})
p = figure(x_range=PCTS, title='1M msg/s x 4: три упаковки (среднее по получателям)',
           height=400, width=760, y_axis_label='наносекунды', toolbar_location='above')
xs = [i + 0.5 for i in range(len(PCTS))]
for off, col, colour in [(-0.25, 'MIN_PACK=4  (999k pkt/s)', GREY),
                         ( 0.00, 'MIN_PACK=8  (500k pkt/s)', BLUE),
                         ( 0.25, 'MIN_PACK=16 (250k pkt/s)', GREEN)]:
    p.vbar(x=[x + off for x in xs], top=cmp[col], width=0.22, color=colour, legend_label=col)
p.yaxis.formatter = NumeralTickFormatter(format='0,0')
p.legend.location = 'top_left'
p.y_range.start = 0
show(p)

print('отрезок a -- хвост и максимум (общий для всех четырёх получателей):')
print(f"  MIN_PACK=4 : p50 {1573:>6,}  p99.999 {23401:>7,}  max        --")
print(f"  MIN_PACK=8 : p50 {4123:>6,}  p99.999 {60436:>7,}  max {207224:>7,}  <- остановки на хосте")
print(f"  MIN_PACK=16: p50 {7406:>6,}  p99.999 {37272:>7,}  max {47268:>7,}")
cmp

**Что видно.** `MIN_PACK=8` лучший по p50, p99 и p99.9: вдвое меньше пакетов, чем
у `MIN_PACK=4`, и на 33–38% лучшая задержка. `a` при этом вырос с 1 573 до 4 123 нс
— ровно по формуле — и всё равно окупился.

**p99.99 у `MIN_PACK=8` — это остановки на хосте, а не упаковка.** Доказывает
третий столбец: у `MIN_PACK=16` отрезок `a` **вдвое больше** (7 406 нс), а p99.99
**вдвое лучше**. Если бы хвост делала упаковка, зависимость была бы обратной.

Смотрите на печать под графиком: `a` max = 207 224 нс у `MIN_PACK=8` против 47 268
у `MIN_PACK=16`. Плюс `c` max = 153 473 и 155 588 нс у двух получателей. Оба
отрезка целиком внутри одного хоста — это producer и consumer, снятые
планировщиком на сотни микросекунд, транспорт здесь ни при чём.

`MIN_PACK=16` даёт ещё вдвое меньше пакетов (250k на четырёх получателей) и самый
чистый хвост, но платит `a` в 7.4 мкс и по p50 проигрывает.

## 6. Ловушка: бюджет ожидания может отменить батчинг

Четыре прогона 600k x 4 с `PACK` = 4, 6, 8 и 12 дали **одинаковые** 2.15 кадра в
датаграмме. Причина — `PACK_WAIT_SPINS=1000`: бюджет истекал раньше, чем
датаграмма наполнялась, и упаковку задавал он, а не `MIN_PACK`.

In [ ]:
p = figure(x_range=list(fanout600.config),
           title='600k msg/s x 4 получателя: три конфигурации',
           height=420, width=760, y_axis_label='наносекунды', toolbar_location='above')

xs = [i + 0.5 for i in range(len(fanout600))]
for off, lo, hi, colour, name in [
    (-0.22, 'p50_lo',  'p50_hi',  BLUE,  'e2e p50'),
    ( 0.00, 'p99_lo',  'p99_hi',  GREEN, 'e2e p99'),
    ( 0.22, 'p999_lo', 'p999_hi', RED,   'e2e p99.9'),
]:
    # столбец от минимума до максимума по четырём получателям
    p.vbar(x=[x + off for x in xs], bottom=fanout600[lo], top=fanout600[hi],
           width=0.2, color=colour, legend_label=name + ' (разброс по получателям)')

p.yaxis.formatter = NumeralTickFormatter(format='0,0')
p.legend.location = 'top_right'
p.y_range.start = 0
show(p)

for _, r in fanout600.iterrows():
    print(f"{r.config.replace(chr(10), ' '):<32} {r.frames_per_dgram:>5.2f} кадр/дгр  "
          f"{r.pps:>9,} pkt/s   a p50 {r.a50:>6,} нс")

**Что видно.** Слева — конфигурация, где `MIN_PACK` не действовал: пакетный поток
встал на 1.11M pkt/s, то есть выше полки шейпера (1.081M), и задержка худшая из
трёх — несмотря на самый маленький `a` (1 961 нс).

Справа — та же нагрузка с рабочим бюджетом: 8 кадров в датаграмме, 300k pkt/s,
`e2e` p50 **13.4–15.3 мкс**. Ожидание в 6 780 нс окупилось втрое.

Отсюда правило, без которого `MIN_PACK` бесполезен:

> **`PACK_WAIT_SPINS` > `MIN_PACK` / `RATE` / 2.24 нс**, с запасом вдвое.

Умолчание 10000 покрывает наполнение до ~22 мкс. Занижать его стоит только как
осознанный потолок на ожидание — и тогда упаковку определяет он, а не `MIN_PACK`.

Побочно этот же набор — ещё одно свидетельство против jumbo: `PACK 12` на MTU 9001
проиграл `PACK 8` на MTU 1500 по всем перцентилям.

## 7. Распределение целиком

Перцентили — сводка, и она скрывает форму. Ячейки ниже читают сырой
`latencies.csv` любого прогона. Путь можно задать переменной окружения
`LATENCIES_CSV`; иначе берётся первый найденный в `harness/results/`.

In [9]:
import os

def find_csv():
    env = os.environ.get('LATENCIES_CSV')
    if env and Path(env).exists():
        return Path(env)
    for base in (Path('harness/results'), Path('results'), Path('.')):
        hits = sorted(base.glob('**/latencies.csv'))
        if hits:
            return hits[-1]
    return None

csv_path = find_csv()
if csv_path is None:
    ns = None
    print('latencies.csv не найден -- графики ниже пропускаются.')
    print('Задайте LATENCIES_CSV=/путь/к/latencies.csv или запустите прогон.')
else:
    df = pd.read_csv(csv_path)
    ns = df['latency_ns'].to_numpy()
    print(f'{csv_path}: {ns.size:,} отсчётов, '
          f'min {ns.min():,} / p50 {int(np.percentile(ns, 50)):,} / max {ns.max():,} нс')
    print('ВНИМАНИЕ: это СЫРАЯ задержка, без поправки часов -- в ней сидит смещение')
    print('между хостами. Форму распределения оно сдвигает целиком, не искажая.')

latencies.csv не найден -- графики ниже пропускаются.
Задайте LATENCIES_CSV=/путь/к/latencies.csv или запустите прогон.


In [10]:
if ns is not None:
    clip = np.percentile(ns, 99.9)
    counts, edges = np.histogram(ns[ns <= clip], bins=80)
    p = fig('Распределение (обрезано по p99.9)', 'наносекунды', 'отсчётов')
    p.quad(top=counts, bottom=0, left=edges[:-1], right=edges[1:],
           fill_color=BLUE, line_color='white', alpha=0.85)
    p.y_range.start = 0
    show(p)

In [11]:
if ns is not None:
    # Хвост: по оси X -- 'расстояние до 100-го перцентиля' в лог-масштабе, чтобы
    # хвост получил реальную ширину. Среднее скрывает его полностью.
    s = np.sort(ns)
    q = np.array([50, 90, 99, 99.9, 99.99, 99.999])
    vals = np.percentile(s, q)
    p = fig('Хвост распределения', '100 - перцентиль (log)', 'наносекунды (log)',
            y_log=True, x_axis_type='log')
    p.line(100 - q, vals, line_width=2.5, color=BLUE)
    p.scatter(100 - q, vals, size=10, color=BLUE)
    p.add_tools(HoverTool(tooltips=[('задержка', '@y{0,0} нс')]))
    show(p)
    for qq, vv in zip(q, vals):
        print(f'  p{qq:<8} {int(vv):>12,} нс')

## Выводы

1. **Шейпит среда пакеты, а не сообщения.** Полка измерена: 1.081M pkt/s на
   пару хостов, разброс 0.6% на восьмикратном диапазоне предложенной нагрузки.
   Пока транспорт слал пакет на сообщение, эта полка была потолком по темпу.

2. **Укладка нескольких кадров в одну датаграмму снимает потолок.** Границы
   кадров восстанавливаются из `frame_size(type)` в заголовке каждого, поэтому
   на провод не добавляется ни байта служебных данных. При 1M msg/s одному
   получателю: 673 мкс и 16.2% потерь → **8.8 мкс и ноль потерь**, 76x.
   На веере: 4 млн сообщений в секунду на четырёх получателей, ноль потерь.

3. **Задержка провода перестала зависеть от темпа сообщений** — только от
   пакетного потока, который батчинг удерживает почти постоянным.

4. **Цена батчинга измерима и предсказуема заранее:** `a` p50 ≈
   `min_pack / (2 x rate)`, проверено на восьми прогонах. Это превращает выбор
   `min_pack` из перебора в расчёт.

5. **Ниже ~200k pkt/s опускаться незачем** — `b` не улучшается, а `a` растёт.
   Правило `min_pack ≈ rate / 200 000` подтверждено предсказанием, которое
   сбылось с запасом.

### Чего здесь нет

Полный список незакрытого — в [SOLUTION.md](harness/SOLUTION.md). Главное:
**поведение под индуцированными потерями не измерено**, хотя формат данных и
`MonotonicGate` под этот сценарий спроектированы. Все замеры — L2, одна
подсеть, инстансы c6in.